# 06 - Walk-forward 'paper trading' simulation

A clean out-of-sample test. Every 21 trading days we re-fit the per-factor weights using
ONLY data observable up to that date, then trade the next 21 days with those weights.

* Warmup: 252 trading days (~1 year)
* Refit cadence: 21 days
* No look-ahead anywhere; every signal at date t was constructible from data up to t-1.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from mfa.config import load_config
from mfa.viz import apply_style
apply_style()

cfg = load_config(ROOT / "configs" / "default.yaml")
print("project:", cfg.project_root.name, "| processed:", cfg.processed_dir.exists())


In [ ]:
from mfa.data import build_panels

def _load_raw_panels():
    out = {}
    for k in ["open","high","low","close","volume","vwap","dollar_volume"]:
        p = cfg.processed_dir / f"panel_{k}.parquet"
        if p.exists():
            df = pd.read_parquet(p)
            df.index = pd.to_datetime(df.index, utc=True)
            out[k] = df
    if not out:
        raise RuntimeError("No cached panels found. Run scripts/01_download_data.py first.")
    return out

def _load_factors():
    d = cfg.processed_dir / "factors"
    out = {}
    for p in sorted(d.glob("*.parquet")):
        df = pd.read_parquet(p)
        df.index = pd.to_datetime(df.index, utc=True)
        out[p.stem] = df
    return out

raw_panels = _load_raw_panels()
benchmark = cfg.section("universe").get("benchmark", "SPY")
panels = build_panels(raw_panels, benchmark=benchmark, min_history_days=252)
factors = _load_factors()
print(f"loaded {len(panels)} panels and {len(factors)} factors")
print("date range:", panels["close"].index.min().date(), "->", panels["close"].index.max().date())
print("universe size:", panels["close"].shape[1], " benchmark:", benchmark)


In [ ]:
from mfa.data import forward_returns
from mfa.evaluation import quantile_returns, long_short_returns
from mfa.evaluation.stats import perf_stats
from mfa.portfolio import factor_timing_weights, long_short_quantile_weights
from mfa.backtest import BacktestConfig, CostModel, simulate_paper_trading

daily_ret = panels["close"].pct_change().fillna(0.0)
bm = panels["benchmark_close"].iloc[:, 0].pct_change().fillna(0.0)
cm = CostModel(commission_bps=1.0, slippage_bps=2.0, borrow_cost_bps_annual=50.0)
config = BacktestConfig(rebalance=5, initial_capital=1_000_000, cost_model=cm, benchmark_returns=bm)

def weight_fn(facts_now, dates_now):
    fl = forward_returns(panels["close"].loc[panels["close"].index <= dates_now[-1]], [1])[1]
    pnl = pd.DataFrame({n: long_short_returns(quantile_returns(f, fl, 5)) for n, f in facts_now.items()})
    return factor_timing_weights(pnl, method="rolling_sharpe", lookback=63)

def sizing_fn(comp):
    return long_short_quantile_weights(comp, top_q=0.2, bot_q=0.2, gross_leverage=2.0, max_weight=0.10)

print("starting walk-forward simulation...")
live = simulate_paper_trading(factors, daily_ret, weight_fn=weight_fn, sizing_fn=sizing_fn,
                              config=config, refit_every=21, warmup=252)
perf = perf_stats(live.backtest.net_returns)
print(f"done. sharpe={perf['sharpe']:+.2f}  ann={perf['ann_return']:+.2%}  mdd={perf['max_drawdown']:.2%}")


## 1. Out-of-sample equity & drawdown

In [ ]:
from mfa.viz import strategy_tearsheet
strategy_tearsheet(live.backtest.net_returns, benchmark=bm, turnover=live.backtest.turnover,
                   perf=perf, title="Walk-forward paper trading -- net of costs");


## 2. Factor weights through time (out-of-sample)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
live.factor_weights_history.plot.area(ax=ax, lw=0, alpha=0.85, colormap="tab20")
ax.set_title("Walk-forward factor weights"); ax.set_ylabel("weight"); ax.set_ylim(0, 1)
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), ncol=1, fontsize=7); plt.tight_layout(); plt.show()


## 3. Top names long / short on the last date

In [ ]:
last_w = live.backtest.weights_held.iloc[-1].sort_values()
top_short = last_w.head(8)
top_long = last_w.tail(8)
df = pd.concat([top_short, top_long])
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ["#E63946" if v < 0 else "#06A77D" for v in df.values]
ax.barh(df.index, df.values, color=colors, edgecolor="black")
ax.axvline(0, color="black", lw=0.7); ax.set_title(f"Latest book on {live.backtest.weights_held.index[-1].date()}")
ax.set_xlabel("weight"); plt.show()


## 4. Year-by-year walk-forward stats

In [ ]:
r = live.backtest.net_returns
def _yr_stats(s: pd.Series) -> dict:
    return {
        "ret": (1+s).prod()-1,
        "vol": s.std()*np.sqrt(252),
        "sharpe": (s.mean()/s.std())*np.sqrt(252) if s.std()>0 else np.nan,
        "max_dd": (((1+s).cumprod() / (1+s).cumprod().cummax()) - 1).min(),
    }
groups = r.groupby(r.index.year)
yearly = pd.DataFrame({y: _yr_stats(g) for y, g in groups}).T
yearly.index.name = "year"
yearly.style.format({"ret":"{:+.2%}","vol":"{:.2%}","sharpe":"{:+.2f}","max_dd":"{:.1%}"}).background_gradient(cmap="RdYlGn", subset=["sharpe","ret"])


### Takeaways

* Walk-forward results are the only ones a real PM should trust.
* Even with realistic costs, the out-of-sample tearsheet exposes regime risk -- some years
  the multi-factor tilt rewards us, others it pays the cost of being wrong.
* Next steps to improve: sector neutralization, beta neutralization, smarter factor timing
  (e.g. macro-conditional weights), and a much wider universe.